In [ ]:
!pip install -q segmentation-models-pytorch albumentations

In [ ]:
import torch, cv2, numpy as np, matplotlib.pyplot as plt
from torchvision import transforms
import segmentation_models_pytorch as smp

model = smp.Unet(encoder_name="resnet34", encoder_weights=None, in_channels=3, classes=1)
model.load_state_dict(torch.load('/content/drive/MyDrive/Colab Notebooks/unet_melanoma_epoch.pth', map_location='cpu'))
model.eval()

def predict_and_show(path):
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    img_r = cv2.resize(img, (256, 256))
    t = transforms.ToTensor()(img_r).unsqueeze(0)
    with torch.no_grad():
        m = (torch.sigmoid(model(t)).squeeze().cpu().numpy() > 0.5).astype(np.uint8)
    c, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    out = img_r.copy()
    cv2.drawContours(out, c, -1, (255, 0, 0), 2)
    plt.figure(figsize=(12,6))
    for i,(im,ttl,cmap) in enumerate([(img_r,"Original",None),(m,"Predicted Mask",'gray'),(out,"Mask Contour",None)],1):
        plt.subplot(1,3,i); plt.title(ttl); plt.imshow(im,cmap=cmap); plt.axis("off")
    plt.tight_layout(); plt.show()

predict_and_show("/content/drive/MyDrive/Colab Notebooks/Dataset/1200px-Melanoma.jpg")

In [ ]:
def extract_asymmetry(mask, visualize=False, threshold=0.2, image_path=None):
    ys, xs = np.where(mask > 0)
    if not len(xs) or not len(ys): return 0
    x0,x1,y0,y1 = np.min(xs), np.max(xs), np.min(ys), np.max(ys)
    t = mask[y0:y1+1, x0:x1+1]
    h,w = t.shape; s = max(h,w)
    p = np.zeros((s,s), np.uint8)
    p[(s-h)//2:(s-h)//2+h, (s-w)//2:(s-w)//2+w] = t
    c = p.shape[1]//2
    L,R = p[:, :c], p[:, c:]
    mw = min(L.shape[1], R.shape[1])
    L,R = L[:, :mw], R[:, :mw]
    aL,aR = np.sum(L>0), np.sum(R>0)
    r = abs(aL - aR) / max(aL, aR) if max(aL,aR) else 0
    a = int(r > threshold)
    if visualize:
        plt.figure(figsize=(10,4))
        plt.suptitle(f"Image Path: {image_path}" if image_path else "Asymmetry Visualization", fontsize=12)
        for i,(img,title) in enumerate([(L,f"Left\n{aL}"),(R,f"Right\n{aR}")],1):
            plt.subplot(1,3,i); plt.imshow(img,cmap='gray'); plt.title(title); plt.axis("off")
        plt.subplot(1,3,3); plt.bar(['Left','Right'],[aL,aR],color=['blue','orange'])
        plt.title(f"Diff Ratio={r:.2f}\nAsymmetry={a}"); plt.ylim(0,max(aL,aR)*1.2)
        plt.tight_layout(rect=[0,0,1,0.9]); plt.show()
    return a

In [7]:
def get_pred_mask(image_path):
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(image, (256, 256))
    input_tensor = transforms.ToTensor()(resized).unsqueeze(0)

    with torch.no_grad():
        output = model(input_tensor)
        pred_mask = torch.sigmoid(output).squeeze().cpu().numpy()
        pred_mask = (pred_mask > 0.5).astype(np.uint8)
    return pred_mask, resized

pred_mask, _ = get_pred_mask(test_image_path)

In [ ]:
asymmetry_result = extract_asymmetry(pred_mask, visualize=True, threshold=0.2, image_path = test_image_path)
print("Asymmetry =", asymmetry_result)


In [ ]:
def extract_border_irregularity_area_based(mask, threshold_ratio=0.02, visualize=False):
    cnts,_=cv2.findContours(mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    if not cnts: return 0
    c=max(cnts,key=cv2.contourArea)
    a_c,a_h=cv2.contourArea(c),cv2.contourArea(cv2.convexHull(c))
    r=(a_h - a_c)/a_h if a_h else 0
    ir=int(r>threshold_ratio)
    if visualize:
        print(f"Area Contour={a_c}, Hull={a_h}, Defect={a_h - a_c}, Ratio={r:.4f} → Irregular={ir}")
        plt.figure(figsize=(12,4))
        plt.subplot(1,3,1); plt.imshow(mask,cmap='gray'); plt.title("Original Mask"); plt.axis('off')
        o=cv2.cvtColor(mask,cv2.COLOR_GRAY2BGR)
        cv2.drawContours(o,[c],-1,(0,255,0),1); cv2.drawContours(o,[cv2.convexHull(c)],-1,(255,0,0),1)
        plt.subplot(1,3,2); plt.imshow(o); plt.title("Green:Contour | Blue:Hull"); plt.axis('off')
        plt.subplot(1,3,3); plt.bar(["Defect Ratio"],[r],color="red" if ir else "green")
        plt.axhline(threshold_ratio,color='blue',ls='--',label=f"Threshold={threshold_ratio}")
        plt.title(f"Irregular={ir} (Ratio={r:.4f})"); plt.ylim(0,max(0.1,r+0.01)); plt.legend()
        plt.tight_layout(); plt.show()
    return ir

In [ ]:
border_irregularity = extract_border_irregularity_area_based(pred_mask, threshold_ratio=0.03, visualize=True)
print("Border Irregularity =", border_irregularity)

In [ ]:
from webcolors import rgb_to_hex

def extract_color_variation(image, n_colors=3, visualize=False):
    p = image.reshape(-1,3)
    k = KMeans(n_clusters=n_colors, random_state=42, n_init=10).fit(p)
    c = k.cluster_centers_.astype(int)
    v = 1 if len(np.unique(k.labels_)) >= 2 else 0
    if visualize:
        plt.figure(figsize=(8,6))
        plt.subplot(121); plt.imshow(image); plt.title("Original Image"); plt.axis('off')
        ci = np.zeros((100,100*n_colors,3),np.uint8)
        for i in range(n_colors): ci[:,i*100:(i+1)*100] = c[i]
        plt.subplot(122); plt.imshow(ci); plt.title("Main Colors"); plt.axis('off')
        plt.show()
        print("Các màu chủ đạo (RGB & Hex):")
        for i,col in enumerate(c): print(f"Màu {i+1}: RGB={col}, Hex={rgb_to_hex(tuple(col))}")
        print(f"→ Số màu khác biệt: {len(np.unique(k.labels_))}\nColor Variation = {v}")
    return v

In [ ]:
pred_mask, image = get_pred_mask(test_image_path)  # Lấy resized image trả về ở đây

color_var = extract_color_variation(image, visualize=True)
